# Bill splitting: thinking, tools, and the policy that chooses actions

The same fixed Gemini model can solve a bill internally, choose a calculator, or be instructed to route arithmetic to that calculator. We compare HIGH/no tool, LOW/optional tool, LOW/explicit tool, and a LOW/no-tool control. This is task-agent improvement, not recursion.

Synthetic stress set: 56 receipts, split 24 development / 32 held-out, two attempts per receipt. Longer group receipts and receipt discounts add ordinary calculation steps; fractional tax rates and uneven splits make precision matter. These are not real restaurant receipts or an estimate of everyday failure frequency. One old failure is a labeled development regression. The other tasks are generated from a fixed seed before running; no failure filtering.


## Recorded results — October 6, 2026

| Policy | Correct | Tool calls | Input | Output | Reported thinking | Total | Median request seconds |
|---|---:|---:|---:|---:|---:|---:|---:|
| LOW_no_tool | 64/64 | 0 | 23,230 | 1,047 | 40,815 | 65,092 | 6.06 |
| HIGH_no_tool | 64/64 | 0 | 23,230 | 955 | 287,357 | 311,542 | 29.11 |
| LOW_optional_tool | 64/64 | 60 | 25,406 | 14,280 | 6,685 | 46,371 | 3.54 |
| LOW_explicit_tool | 64/64 | 64 | 27,262 | 14,456 | 457 | 42,175 | 3.05 |

LOW/no-tool made **one arithmetic error in 48 development attempts**: USD 194.22 instead of USD 185.43 on an 18-line receipt. It passed all held-out attempts. The stress set did not establish a held-out accuracy gain or demonstrate frequent model failure. The fixed improver voluntarily chose a tool; both tool arms use that same validated function. The explicit delegation instruction is a manually specified comparison policy.

Explicit delegation saved **9.0% total tokens versus optional tool use**, **86.5% versus HIGH/no-tool**, and **35.2% versus LOW/no-tool** on held-out receipts. All policies passed 64/64. The first contrast fixes LOW and the tool; the HIGH contrast changes both compute allocation and tool access. This is not an equal-compute experiment.

Creation cost **21,758 tokens**. Observed savings versus LOW/no-tool repay creation alone after about **61 receipts**, or **153** including the explicit candidate’s development evaluation. This assumes average savings continue and excludes the other research comparisons. Total experiment: **449 calls, 884,353 reported tokens**; no request/execution failures and complete total-token usage coverage.

Some LOW responses omit the thinking-count field. The sums above are reported counts; raw omissions remain null. For these responses, total minus input minus output reconciles with reported thinking or zero when the field is omitted. The local function uses CPU time, not Gemini tokens. No second synthesis call is included. Latency is measured request time; concurrent batches and API variability affect it.

The 11% tax / 15% tip group was requested while the main run was active. Its eight receipts were generated from seed 2026100603, then evaluated in an independent batch with the policies frozen. All four policies passed its 16 attempts. The rerunnable source folds this group into held-out evaluation. Raw records preserve the actual execution protocol. The fixed original stress seed is 2026100602. No failures were filtered into either set.

The improver’s rationale claimed guarantees and token savings before they were tested. Those claims are not evaluation results. Read the actual traces. The improver was never edited; recursion remains deferred. Configuration: model `gemini-3.8-flash`, SDK 2.28.0, output cap 32,768, timeout 180 seconds, three workers per batch, no retries. [Google thinking docs](https://ai.google.dev/gemini-api/docs/thinking).


In [ ]:
%pip -q install google-genai==2.28.0

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


## 1. Freeze the tasks and collect LOW baseline responses

The evaluator uses rational Fraction arithmetic independently of the tool’s Decimal implementation. All policies receive explicit discount, tip and rounding rules. We removed the earlier prose-calculation requirement: tool calls can route arguments without also narrating a calculation. Input, final-output, thinking and total tokens are logged separately, including incomplete responses when usage is returned. Missing metadata stays null.


In [4]:
import os, ast, json, random, time, statistics
from decimal import Decimal, ROUND_HALF_UP
from fractions import Fraction
from concurrent.futures import ThreadPoolExecutor
from google import genai
from google.genai import types
from IPython.display import JSON, display

MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
REPEATS, WORKERS = 2, 3
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"], vertexai=False,
    http_options=types.HttpOptions(timeout=180000,
        retry_options=types.HttpRetryOptions(attempts=1)))
calls=[]

def ask(prompt, schema, level="LOW"):
    started=time.monotonic()
    record={"thinking_level":level}
    calls.append(record)
    try:
        response=client.models.generate_content(model=MODEL,contents=prompt,
            config={"temperature":0,"max_output_tokens":32768,
                "thinking_config":{"thinking_level":level},
                "response_mime_type":"application/json","response_json_schema":schema,
                "automatic_function_calling":{"disable":True}})
        u=response.usage_metadata
        record.update(prompt_tokens=u.prompt_token_count,output_tokens=u.candidates_token_count,
            thinking_tokens=u.thoughts_token_count,total_tokens=u.total_token_count,
            cached_tokens=u.cached_content_token_count,model_version=response.model_version,
            finish_reason=str(response.candidates[0].finish_reason) if response.candidates else "no candidate")
        if not response.candidates or response.candidates[0].finish_reason!="STOP":
            raise RuntimeError("Incomplete response: "+record['finish_reason'])
        return json.loads(response.text),record
    except Exception as error:
        record.update(error_type=type(error).__name__,status=getattr(error,"code",None))
        error.resources=record
        raise
    finally:record['seconds']=round(time.monotonic()-started,3)

def oracle(task):
    gross=sum((Fraction(x['unit_price'])*x['quantity'] for x in task['items']),Fraction(0))
    net=gross*(1-Fraction(task['discount'])/100)
    taxed=net*(1+Fraction(task['tax'])/100)
    basis=net if task['tip_on']=='subtotal' else taxed
    cents=(taxed+basis*Fraction(task['tip'])/100)/task['diners']*100
    rounded=(2*cents.numerator+cents.denominator)//(2*cents.denominator)
    return Decimal(rounded)/100

def dataset():
    rng=random.Random(2026100602)
    dev,test=[],[]
    for group in ('fractional_rates','itemized_receipts','receipt_discounts'):
        for i in range(16):
            count=1 if group=='fractional_rates' else rng.randint(12,20)
            items=[{'unit_price':str(Decimal(rng.randint(10001,800000) if count==1 else rng.randint(149,7999))/100),
                    'quantity':1 if count==1 else rng.randint(1,6)} for _ in range(count)]
            task={'id':f'{group}-{i:02}', 'group':group, 'items':items,
                'discount':rng.choice(['5','10','12.5','15']) if group=='receipt_discounts' else '0',
                'tax':rng.choice(['7.25','8.875','9.125']),
                'tip':rng.choice(['17.5','18.25','19.625','20']),
                'diners':rng.choice([3,7,11,13,14,17,19,23]),
                'tip_on':'subtotal' if i%2==0 else 'total'}
            # A previously observed error is a named development regression, not held-out evidence.
            if group=='fractional_rates' and i==0:
                task.update(items=[{'unit_price':'3456.37','quantity':1}],discount='0',tax='9.125',tip='19.625',diners=14,tip_on='subtotal')
            (dev if i<8 else test).append(task)
    return dev,test

# Requested during the run, generated before observing any of these cases.
# The policies remain frozen; these are additional held-out receipts.
def ordinary_rates():
    rng=random.Random(2026100603)
    return [{'id':f'ordinary_rates-{i:02}','group':'ordinary_rates',
        'items':[{'unit_price':str(Decimal(rng.randint(199,5999))/100),'quantity':rng.randint(1,4)}
                 for _ in range(rng.randint(8,16))],
        'discount':'10' if i%3==0 else '0','tax':'11','tip':'15',
        'diners':rng.choice([3,7,11,13,17]),'tip_on':'subtotal' if i%2==0 else 'total'}
        for i in range(8)]

development,held_out=dataset()
held_out+=ordinary_rates()
assert not {t['id']for t in development}&{t['id']for t in held_out}
assert oracle(development[0])==Decimal('317.86')
print(MODEL,'|',len(development),'development |',len(held_out),'held-out | seed 2026100602')

ITEM={'type':'object','properties':{'unit_price':{'type':'string'},'quantity':{'type':'integer'}},
    'required':['unit_price','quantity'],'additionalProperties':False}
PARAMS={'items':{'type':'array','items':ITEM},'discount':{'type':'string'},
    'tax':{'type':'string'},'tip':{'type':'string'},'diners':{'type':'integer'},
    'tip_on':{'type':'string','enum':['subtotal','total']}}
ANSWER={'type':'object','properties':{'action':{'type':'string','enum':['answer','tool']},
    'amount':{'type':'string'},'arguments':{'type':'object','properties':PARAMS,
        'required':list(PARAMS),'additionalProperties':False}},
    'required':['action','amount'],'additionalProperties':False}

BASELINE="Calculate the bill accurately and answer the user's request."

def trajectory(job):
    instruction,level,tool,task,repeat=job
    contract=("receipt_split(items, discount, tax, tip, diners, tip_on) is available. "
        "You may calculate internally or return action='tool' with arguments and an empty amount. "
        "The host executes those arguments and uses the tool result as the final answer." if tool else
        "No tools are available. Compute internally and return action='answer' with the decimal dollar amount.")
    prompt=(instruction+'\n'+contract+
        "\nEach item has a dollar unit_price and whole quantity. Sum price times quantity. "
        "Apply the receipt discount to that subtotal BEFORE tax. Compute tax on the discounted subtotal. "
        "Tip applies to the discounted subtotal when tip_on='subtotal', or the discounted total AFTER tax when tip_on='total'. "
        "Split equally among diners. Keep full intermediate precision and round ONLY final per-person cents using ROUND_HALF_UP. "
        "No prose calculation is requested.\nReceipt: "+json.dumps({k:task[k]for k in PARAMS}))
    row={'id':task['id'],'group':task['group'],'repeat':repeat,'task':task,'reference':str(oracle(task)),
        'thinking_level':level,'tool_calls':0}
    try:
        answer,resources=ask(prompt,ANSWER,level)
        row.update(resources,response=answer)
        if answer['action']=='tool':
            row['tool_calls']=1
            if not tool:raise ValueError('Tool unavailable')
            args=answer['arguments']
            # Fixed input bounds keep the generated arithmetic grammar inspectable.
            if len(args['items'])>20 or not 1<=args['diners']<=100:
                raise ValueError('Invalid tool bounds')
            if any(type(x['quantity'])is not int or not 1<=x['quantity']<=100 for x in args['items']):
                raise ValueError('Invalid quantity')
            if any(len(str(x))>40 for x in [args['discount'],args['tax'],args['tip']]+[i['unit_price']for i in args['items']]):
                raise ValueError('Numeric input too long')
            start=time.monotonic()
            amount=tool(*(args[k]for k in PARAMS))
            row.update(tool_result=str(amount),tool_seconds=time.monotonic()-start)
        else:amount=Decimal(answer['amount'].strip().removeprefix('USD').strip().removeprefix('$').replace(',',''))
        row.update(amount=str(amount),correct=amount==oracle(task))
    except Exception as error:
        row.update(getattr(error,'resources',{}),correct=False,error_type=type(error).__name__,
            error_detail=str(error) if isinstance(error,(RuntimeError,ValueError,KeyError)) else None)
    return row

def evaluate(instruction,level,tool,tasks):
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        jobs=[(instruction,level,tool,t,r)for t in tasks for r in range(REPEATS)]
        rows=[]
        for row in pool.map(trajectory,jobs):
            rows.append(row)
            if len(rows)==len(jobs)//2:print('Half the attempts completed:',len(rows),'/',len(jobs),flush=True)
        return rows

def summary(rows):
    reported=[r for r in rows if r.get('total_tokens')is not None]
    completed=[r for r in rows if 'error_type'not in r]
    return {'correct':sum(r['correct']for r in rows),'attempts':len(rows),
        'task_errors':sum(not r['correct'] and 'error_type'not in r for r in rows),
        'failed_requests_or_execution':sum('error_type'in r for r in rows),
        'tool_calls':sum(r['tool_calls']for r in rows),
        **{k:sum(r[k]or 0 for r in reported)for k in ('prompt_tokens','output_tokens','thinking_tokens','total_tokens')},
        'usage_reported':len(reported),'thinking_count_reported':sum(r.get('thinking_tokens')is not None for r in reported),
        'median_successful_request_seconds':round(statistics.median([r['seconds']for r in completed]),2) if completed else None}

baseline_dev=evaluate(BASELINE,'LOW',None,development)
print('LOW no-tool development:',summary(baseline_dev))
for row in baseline_dev:
    if not row['correct']:print('Failure:',row['id'],row['repeat'],row.get('amount'),row['reference'],row.get('error_type'))


gemini-3.8-flash | 24 development | 24 held-out | seed 2026100602
LOW no-tool development: {'correct': 47, 'attempts': 48, 'task_errors': 1, 'failed_requests_or_execution': 0, 'tool_calls': 0, 'prompt_tokens': 18000, 'output_tokens': 783, 'thinking_tokens': 31736, 'total_tokens': 50519, 'usage_reported': 48, 'thinking_count_reported': 47, 'median_successful_request_seconds': 6.01}
Failure: itemized_receipts-01 0 194.22 185.43 None


## 2. Propose an improvement, then compare policies

The fixed improver sees development traces only and may choose a tool, instructions or no change. Both tool arms use the identical validated calculator. Optional tool use leaves the original instruction intact; the explicit policy tells the agent to route the arithmetic and stop calculating itself. Both use LOW. HIGH/no-tool changes the thinking setting; LOW/no-tool is its control. The tiny syntax validator is an arithmetic boundary, not a general sandbox.


In [6]:
def compile_tool(source):
    tree=ast.parse(source)
    allowed=(ast.Module,ast.FunctionDef,ast.arguments,ast.arg,ast.Assign,ast.Return,
        ast.Expr,ast.Name,ast.Load,ast.Store,ast.Constant,ast.BinOp,ast.Add,ast.Sub,
        ast.Mult,ast.Div,ast.Call,ast.Attribute,ast.keyword,ast.Subscript,
        ast.GeneratorExp,ast.comprehension,ast.IfExp,ast.Compare,ast.Eq)
    if len(source)>3000 or len(list(ast.walk(tree)))>260 or len(tree.body)!=1:
        raise ValueError('Tool exceeds arithmetic grammar')
    fn=tree.body[0]
    if not isinstance(fn,ast.FunctionDef) or fn.name!='receipt_split' or fn.decorator_list:
        raise ValueError('Expected receipt_split')
    if fn.args.defaults or fn.args.kw_defaults or fn.args.vararg or fn.args.kwarg:
        raise ValueError('No default or variadic arguments')
    for node in ast.walk(tree):
        if not isinstance(node,allowed):raise ValueError('Disallowed syntax: '+type(node).__name__)
        if isinstance(node,(ast.Name,ast.arg)) and '__'in getattr(node,'id',getattr(node,'arg','')):raise ValueError('Disallowed name')
        if isinstance(node,ast.Attribute) and node.attr!='quantize':raise ValueError('Disallowed attribute')
        if isinstance(node,ast.Call):
            if isinstance(node.func,ast.Name) and node.func.id not in ('Decimal','sum','str','int'):raise ValueError('Disallowed call')
            if not isinstance(node.func,(ast.Name,ast.Attribute)):raise ValueError('Disallowed call target')
        if isinstance(node,ast.comprehension) and (node.is_async or node.ifs or not isinstance(node.iter,ast.Name) or node.iter.id!='items'):
            raise ValueError('Generators only over bounded input items')
        if isinstance(node,ast.Constant) and len(str(node.value))>100:raise ValueError('Large constant')
    scope={'__builtins__':{},'Decimal':Decimal,'sum':sum,'str':str,'int':int,'ROUND_HALF_UP':ROUND_HALF_UP}
    exec(compile(tree,'generated_receipt_tool','exec'),scope)
    return scope['receipt_split']

CHANGE={'type':'object','properties':{'change':{'type':'string','enum':['no_change','instruction','tool']},
    'instruction':{'type':'string'},'tool_code':{'type':'string'},'reason':{'type':'string'}},
    'required':['change','instruction','tool_code','reason'],'additionalProperties':False}
TOOL_SPEC="""A tool must be one function receipt_split(items, discount, tax, tip, diners, tip_on).
items is a bounded list of dicts with dollar string unit_price and integer quantity.
Apply discount to the summed subtotal before tax; tip basis is discounted subtotal
for 'subtotal' or discounted after-tax total for 'total'. Return Decimal final
per-person dollars, rounding only final cents with ROUND_HALF_UP. Provided:
Decimal, sum, str, int, ROUND_HALF_UP. Allowed: assignment, arithmetic, subscripting,
sum with generator expressions directly over items, comparisons, conditional
expressions, Decimal.quantize. No imports, loops, annotations, defaults,
decorators, extra functions or markdown fences. Code under 3000 characters.
Do not hardcode prices or case answers."""
IMPROVER="""Improve a bill task agent using the supplied development trajectories.
You remain fixed. You cannot change model, objective, data, evaluator or harness.
Choose no_change, an instruction change or a reusable tool. Do not invent failures.
Objective: exact cents on future tasks, with total model tokens a secondary cost.
A tool is optional. Keep instructions under 100 words and rationale under 100 words.
For no_change preserve the original instruction. Non-tool proposals use empty code.
"""+TOOL_SPEC
proposal,proposal_usage=ask(IMPROVER+'\nCurrent instruction: '+BASELINE+
    '\nDevelopment trajectories: '+json.dumps(baseline_dev),CHANGE)
print('Autonomous proposal:',proposal)
# Policy comparisons require the SAME tool in both tool arms. If the improver
# chooses no tool, request one explicitly and label that as a directed ablation.
directed_creation=False
creation_usage=proposal_usage
if proposal['change']=='tool':source=proposal['tool_code']
else:
    directed_creation=True
    directed,creation_usage=ask('Write the following tool for a controlled ablation. '+TOOL_SPEC,
        {'type':'object','properties':{'tool_code':{'type':'string'}},'required':['tool_code'],'additionalProperties':False})
    source=directed['tool_code']
tool=compile_tool(source)
for task in development:
    assert tool(*(task[k]for k in PARAMS))==oracle(task),'Generated tool failed development contract'
print('Generated tool passed all development checks. Creation usage:',creation_usage)

POLICIES={
    'LOW_no_tool':(BASELINE,'LOW',None),
    'HIGH_no_tool':(BASELINE,'HIGH',None),
    'LOW_optional_tool':(BASELINE,'LOW',tool),
    'LOW_explicit_tool':('Use receipt_split to solve every bill. Copy the receipt arguments accurately. Do not recompute arithmetic yourself. Return action=tool, the arguments, and an empty amount; the host executes the calculator.','LOW',tool)}
dev_rows={'LOW_no_tool':baseline_dev}
for name in ('HIGH_no_tool','LOW_optional_tool','LOW_explicit_tool'):
    dev_rows[name]=evaluate(*POLICIES[name],development)
    print('Development',name,summary(dev_rows[name]))
# Freeze selection before evaluating any held-out case: accuracy first, then
# total tokens on a tie. Failures disqualify a candidate from retention.
retained='LOW_no_tool'
if not any('error_type'in r for r in baseline_dev):
    for name,rows in dev_rows.items():
        if any('error_type'in r for r in rows):continue
        a,b=summary(rows),summary(dev_rows[retained])
        if (a['correct'],-a['total_tokens'])>(b['correct'],-b['total_tokens']):retained=name
print('Retained on development:',retained)


Autonomous proposal: {'change': 'tool', 'instruction': 'Use receipt_split to calculate the split bill accurately and return the amount.', 'tool_code': "def receipt_split(items, discount, tax, tip, diners, tip_on):\n    subtotal = sum(Decimal(item['unit_price']) * item['quantity'] for item in items)\n    discounted = subtotal * (Decimal('1') - Decimal(str(discount)) / Decimal('100'))\n    tax_amt = discounted * (Decimal(str(tax)) / Decimal('100'))\n    tip_base = discounted if tip_on == 'subtotal' else (discounted + tax_amt)\n    tip_amt = tip_base * (Decimal(str(tip)) / Decimal('100'))\n    total = discounted + tax_amt + tip_amt\n    return (total / Decimal(str(diners))).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)", 'reason': 'An itemized calculation failed due to manual arithmetic errors without thinking tokens. Implementing receipt_split guarantees exact cent precision and saves thinking tokens.'}
Generated tool passed all development checks. Creation usage: {'thinking_level': 

## 3. Test the frozen policies on held-out receipts

Selection uses development accuracy first and tokens only on an accuracy tie. Held-out results never select or revise a policy. HTTP, timeout, incomplete-generation and execution failures are separated from wrong completed answers. Token totals with missing usage do not represent equal completed work. Local tool execution consumes CPU time, not Gemini tokens, and its result is used directly without another model call. A conversational agent would pay additional synthesis costs.


In [8]:
held_rows={}
for name,policy in POLICIES.items():
    held_rows[name]=evaluate(*policy,held_out)
    print('HELD OUT',name,summary(held_rows[name]))
    for group in ('fractional_rates','itemized_receipts','receipt_discounts','ordinary_rates'):
        rows=[r for r in held_rows[name]if r['group']==group]
        print(' ',group,summary(rows))
experiment={'model':MODEL,'seed':2026100602,'repeats':REPEATS,
    'configuration':{'workers':WORKERS,'temperature':0,'max_output_tokens':32768,'timeout_seconds':180,'retries':0},
    'dataset':{'development':development,'held_out':held_out},
    'proposal':proposal,'proposal_usage':proposal_usage,'directed_tool_creation':directed_creation,
    'tool_code':source,'creation_usage':creation_usage,'retained_on_development':retained,
    'development':dev_rows,'held_out':held_rows,'calls':calls,
    'summary':{k:summary(v)for k,v in held_rows.items()}}
print('Total calls:',len(calls),'| reported tokens:',sum(c.get('total_tokens')or 0 for c in calls))
display(JSON(experiment,expanded=False))


FINAL HELD-OUT COMPARISON (including 11% / 15% group)
LOW_no_tool {"correct": 64, "attempts": 64, "task_errors": 0, "failed_requests_or_execution": 0, "tool_calls": 0, "prompt_tokens": 23230, "output_tokens": 1047, "thinking_tokens": 40815, "total_tokens": 65092, "usage_reported": 64, "thinking_count_reported": 64, "median_successful_request_seconds": 6.06}
HIGH_no_tool {"correct": 64, "attempts": 64, "task_errors": 0, "failed_requests_or_execution": 0, "tool_calls": 0, "prompt_tokens": 23230, "output_tokens": 955, "thinking_tokens": 287357, "total_tokens": 311542, "usage_reported": 64, "thinking_count_reported": 64, "median_successful_request_seconds": 29.11}
LOW_optional_tool {"correct": 64, "attempts": 64, "task_errors": 0, "failed_requests_or_execution": 0, "tool_calls": 60, "prompt_tokens": 25406, "output_tokens": 14280, "thinking_tokens": 6685, "total_tokens": 46371, "usage_reported": 64, "thinking_count_reported": 33, "median_successful_request_seconds": 3.54}
LOW_explicit_tool 

Full frozen-policy comparison and all usage records.

## Read the comparison

A good tool does not guarantee a good action policy. Check tool-call counts alongside accuracy, thinking tokens, total tokens and creation cost. The three named policies change both compute allocation and action choice; they are not equal-compute comparisons. There are only 32 distinct held-out receipts. Repeats reveal variability, not new tasks. The old [LOW pilot](https://colab.research.google.com/github/weilinear/rsi_labs/blob/main/trajectory_improvement_low_thinking.ipynb) remains available.
